In [2]:
import numpy as np
W=np.arange(21).reshape(7,3)
print(W)

[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]
 [12 13 14]
 [15 16 17]
 [18 19 20]]


In [3]:
W[2]

array([6, 7, 8])

In [4]:
W[5]

array([15, 16, 17])

In [6]:
idx=np.array([1,3,5])
W[idx]

array([[ 3,  4,  5],
       [ 9, 10, 11],
       [15, 16, 17]])

In [8]:
# embed = Embedding(W)                   
# target_W = embed.forward(idx)          
# out = np.sum(target_W * h, axis=1)

NameError: name 'h' is not defined

In [9]:
import numpy as np

# 정수 무작위 샘플링
print(np.random.choice(10))  # 예: 7
print(np.random.choice(10))  # 예: 3

# 단어 리스트에서 무작위 샘플링
words = ['you', 'say', 'goodbye', 'I', 'hello', '.']
print(np.random.choice(words))  # 예: 'you'

# 여러 개 샘플링 (중복 허용)
print(np.random.choice(words, size=5))

# 여러 개 샘플링 (중복 비허용)
print(np.random.choice(words, size=5, replace=False))

3
3
I
['goodbye' '.' 'you' 'say' 'goodbye']
['.' 'goodbye' 'hello' 'say' 'you']


In [10]:
words = ['you', 'say', 'goodbye', 'I', 'hello', '.']
p = [0.5, 0.1, 0.05, 0.2, 0.05, 0.1]  # 단어별 확률 분포

sample = np.random.choice(words, p=p)
print(sample)

you


In [12]:
# 기존 확률 분포
p = np.array([0.7, 0.29, 0.01])

# 0.75 제곱 보정
new_p = np.power(p, 0.75)
new_p /= np.sum(new_p)  # 정규화

print(new_p)  # 보정된 확률 분포

[0.64196878 0.33150408 0.02652714]


### 자연어처리1 5주차 과제

In [13]:
# coding: utf-8
import sys
sys.path.append('..')
from common.np import *  # import numpy as np
from common.layers import Embedding, SigmoidWithLoss
import collections


class EmbeddingDot:
    def __init__(self, W):
        self.embed = Embedding(W)
        self.params = self.embed.params
        self.grads = self.embed.grads
        self.cache = None

    def forward(self, h, idx):
        target_W = self.embed.forward(idx)
        out = np.sum(target_W * h, axis=1)

        self.cache = (h, target_W)
        return out

    def backward(self, dout):
        h, target_W = self.cache
        dout = dout.reshape(dout.shape[0], 1)

        dtarget_W = dout * h
        self.embed.backward(dtarget_W)
        dh = dout * target_W
        return dh


class UnigramSampler:
    def __init__(self, corpus, power, sample_size):
        self.sample_size = sample_size
        self.vocab_size = None
        self.word_p = None

        counts = collections.Counter()
        for word_id in corpus:
            counts[word_id] += 1

        vocab_size = len(counts)
        self.vocab_size = vocab_size

        self.word_p = np.zeros(vocab_size)
        for i in range(vocab_size):
            self.word_p[i] = counts[i]

        self.word_p = np.power(self.word_p, power)
        self.word_p /= np.sum(self.word_p)

    def get_negative_sample(self, target):
        batch_size = target.shape[0]

        if not GPU:
            negative_sample = np.zeros((batch_size, self.sample_size), dtype=np.int32)

            for i in range(batch_size):
                p = self.word_p.copy()
                target_idx = target[i]
                p[target_idx] = 0
                p /= p.sum()
                negative_sample[i, :] = np.random.choice(self.vocab_size, size=self.sample_size, replace=False, p=p)
        else:
            # GPU(cupy）로 계산할 때는 속도를 우선한다.
            # 부정적 예에 타깃이 포함될 수 있다.
            negative_sample = np.random.choice(self.vocab_size, size=(batch_size, self.sample_size),
                                               replace=True, p=self.word_p)

        return negative_sample



In [14]:

class NegativeSamplingLoss:
    def __init__(self, W, corpus, power=0.75, sample_size=5):
        self.sample_size = sample_size
        self.sampler = UnigramSampler(corpus, power, sample_size)
        self.loss_layers = [SigmoidWithLoss() for _ in range(sample_size + 1)]
        self.embed_dot_layers = [EmbeddingDot(W) for _ in range(sample_size + 1)]

        self.params, self.grads = [], []
        for layer in self.embed_dot_layers:
            self.params += layer.params
            self.grads += layer.grads

    def forward(self, h, target):
        batch_size = target.shape[0]
        negative_sample = self.sampler.get_negative_sample(target)

        # 긍정적 예 순전파
        score = self.embed_dot_layers[0].forward(h, target)
        correct_label = np.ones(batch_size, dtype=np.int32)
        loss = self.loss_layers[0].forward(score, correct_label)

        # 부정적 예 순전파
        negative_label = np.zeros(batch_size, dtype=np.int32)
        for i in range(self.sample_size):
            negative_target = negative_sample[:, i]
            score = self.embed_dot_layers[1 + i].forward(h, negative_target)
            loss += self.loss_layers[1 + i].forward(score, negative_label)

        return loss

    def backward(self, dout=1):
        dh = 0
        for l0, l1 in zip(self.loss_layers, self.embed_dot_layers):
            dscore = l0.backward(dout)
            dh += l1.backward(dscore)

        return dh


In [15]:
import numpy as np

# 테스트를 위한 간단한 말뭉치 데이터 생성
corpus = np.array([0, 1, 2, 3, 4, 1, 2, 3, 1, 2, 3, 4, 0, 1, 2, 3])

# 임베딩 차원과 어휘 크기 설정
vocab_size = 5
embedding_size = 100

# 임베딩 가중치 행렬 초기화
W = 0.01 * np.random.randn(vocab_size, embedding_size)

# NegativeSamplingLoss 인스턴스 생성
negative_sampling_loss = NegativeSamplingLoss(W, corpus, power=0.75, sample_size=2)

# 테스트용 미니배치 데이터 생성
batch_size = 3
h = 0.1 * np.random.randn(batch_size, embedding_size)  # 은닉층의 출력
target = np.array([1, 3, 4])  # 목표 단어의 인덱스

# 순전파
loss = negative_sampling_loss.forward(h, target)
print('손실값:', loss)

# 역전파
dh = negative_sampling_loss.backward()
print('은닉층에 대한 기울기 shape:', dh.shape)
print('첫 번째 임베딩 레이어의 가중치 기울기 shape:', negative_sampling_loss.grads[0].shape)

손실값: 2.076914684682901
은닉층에 대한 기울기 shape: (3, 100)
첫 번째 임베딩 레이어의 가중치 기울기 shape: (5, 100)


#### 알고리즘 수업 코드

In [ ]:
def linear_search(arr, value):
    for item in arr:
        if item == value:
            return True
    return False

def binary_search(arr, value):
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == value:
            return mid
        elif arr[mid] < value:
            left = mid + 1
        else:
            right = mid - 1
    return -1

def main():
    data = []

    print("정수를 계속 입력 받습니다. (0 입력 시 종료)")

    while True:
        try:
            num = int(input("정수 입력: "))
        except ValueError:
            continue

        if num == 0:
            break

        if linear_search(data, num):
            print("중복값이 있습니다")
            continue

        data.append(num)
        data.sort()

    print(f"총 {len(data)}개의 정수가 저장되었습니다.")

    while True:
        try:
            target = int(input("음수 입력 시 절댓값 탐색: "))
        except ValueError:
            continue

        if target >= 0:
            continue

        abs_val = abs(target)
        idx = binary_search(data, abs_val)

        if idx != -1:
            print(f"{abs_val} 값은 인덱스 {idx} 위치에 있습니다.")
        else:
            print("찾는 값이 없습니다")

if __name__ == "__main__":
    main()

정수를 계속 입력 받습니다. (0 입력 시 종료)
중복값이 있습니다
중복값이 있습니다
중복값이 있습니다
중복값이 있습니다
중복값이 있습니다
중복값이 있습니다
중복값이 있습니다
중복값이 있습니다


In [2]:
import sys
sys.path.append('..')  # 상위 디렉터리 모듈 불러오기 위한 설정
from common.util import most_similar, analogy
import pickle

# 저장된 학습 파라미터 파일 불러오기
pkl_file = 'source/deep-learning-from-scratch-2/ch04/cbow_params.pkl'  # 또는 'skipgram_params.pkl'

with open(pkl_file, 'rb') as f:
    params = pickle.load(f)
    word_vecs = params['word_vecs']
    word_to_id = params['word_to_id']
    id_to_word = params['id_to_word']

# 가장 비슷한 단어(top-N) 찾기
querys = ['you', 'year', 'car', 'toyota']
for query in querys:
    most_similar(query, word_to_id, id_to_word, word_vecs, top=5)


[query] you
 we: 0.6103515625
 someone: 0.59130859375
 i: 0.55419921875
 something: 0.48974609375
 anyone: 0.47314453125

[query] year
 month: 0.71875
 week: 0.65234375
 spring: 0.62744140625
 summer: 0.6259765625
 decade: 0.603515625

[query] car
 luxury: 0.497314453125
 arabia: 0.47802734375
 auto: 0.47119140625
 disk-drive: 0.450927734375
 travel: 0.4091796875

[query] toyota
 ford: 0.55078125
 instrumentation: 0.509765625
 mazda: 0.49365234375
 bethlehem: 0.47509765625
 nissan: 0.474853515625
